# Comprensión y calidad inicial de los datos

## Objetivo

Construir evidencia reproducible para documentar las 16 variables de los
datasets y evaluar posteriormente su calidad inicial.

Esta fase distingue entre:

- tipo físico inferido por Pandas;
- significado lógico de la variable;
- función analítica;
- unidad o formato;
- disponibilidad por ciudad;
- definiciones verificadas y ambigüedades.

No se limpian, transforman ni concatenan los CSV originales.

## Fuentes

1. Evidencia observada en los seis CSV versionados.
2. Diccionario y supuestos publicados por
   [Inside Airbnb](https://insideairbnb.com/data-assumptions/).
3. Documentación oficial de Airbnb para conceptos de negocio.

Inside Airbnb es una fuente independiente y no está respaldada oficialmente
por Airbnb. Las definiciones se contrastarán con los datos reales.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_directory = project_root / "data" / "raw" / "airbnb"
csv_files = sorted(data_directory.glob("*csv"))

assert len(csv_files) == 6

datasets = {
    csv_file.name: pd.read_csv(
        csv_file,
        low_memory=False,
    )
    for csv_file in csv_files
}

print(f"Datasets cargados: {len(datasets)}")

Datasets cargados: 6


## 1. Identificadores y variables descriptivas

Se examinan `id`, `host_id`, `name` y `host_name` antes de documentar su
significado y función analítica.

In [2]:
first_variable_group = [
    "id",
    "host_id",
    "name",
    "host_name",
]

variable_profile_records = []

for file_name, dataframe in datasets.items():
    for variable_name in first_variable_group:
        column = dataframe[variable_name]

        variable_profile_records.append(
            {
                "file_name": file_name,
                "variable_name": variable_name,
                "pandas_dtype": str(column.dtype),
                "non_null_percentage": round(
                    column.notna().mean() * 100,
                    2,
                ),
                "distinct_values": int(
                    column.nunique(dropna=True)
                ),
                "sample_values": (
                    column.dropna()
                    .astype(str)
                    .drop_duplicates()
                    .head(3)
                    .tolist()
                ),
            }
        )

first_group_profile = pd.DataFrame(
    variable_profile_records
).sort_values(
    ["variable_name", "file_name"],
    ignore_index=True,
)

display(first_group_profile)

,file_name,variable_name,pandas_dtype,non_null_percentage,distinct_values,sample_values
0,NY_airbnb.csv,host_id,int64,100.00,37457,"[2787, 2845, 4632]"
1,london_airbnb.csv,host_id,int64,100.00,53476,"[43039, 54730, 491286]"
2,madrid_airbnb.csv,host_id,int64,100.00,11325,"[13660, 83531, 82175]"
3,milan_airbnb.csv,host_id,int64,100.00,12213,"[13822, 95941, 121663]"
4,sydney_airbnb.csv,host_id,int64,100.00,27219,"[17061, 55948, 59850]"
5,tokyo_airbnb.csv,host_id,int64,100.00,2954,"[151977, 964081, 341577]"
6,NY_airbnb.csv,host_name,str,99.96,11452,"[John, Jennifer, Elisabeth]"
7,london_airbnb.csv,host_name,str,99.99,14573,"[Adriano, Alina, Chil]"
8,madrid_airbnb.csv,host_name,str,97.31,3900,"[Simon, Abdel, Jesus]"
9,milan_airbnb.csv,host_name,str,99.32,2917,"[Francesca, Jeremy, Marta]"
